# Bronze Ingestion - Healthcare Lakehouses
Este notebook realiza a ingestão dos arquivos brutos da camada **landing** para a camada **bronze**.

## Objetivos
- Criar tabelas Delta na camada bronze
- carregar arquivos brutos apartir dos volumes de landing
- preservar dados o mais próximo possível da origem
- validar a integridade mínima da ingestão

## Estratégia de ingestão
### COPY INTO
Utilizado para arquivos tabulares simples e ingestão incremental idempotente:
- CSV
- CSV staging

### PySpark
Utilizado para casos específicos em que COPY INTO não é melhor opção:
- JSONL
- Parquet com particularidade de tipo
- arquivos previamente normalizados

## Observação
Na camada bronze os campos são mantidos na sua maioria como STRING, evitando conflitos de schema durante a ingestão. A tipagem e validação de negócio serão feitas na camada silver.

##Contexto

In [0]:
%sql
USE CATALOG healthcare_dev;
USE SCHEMA bronze;

SELECT current_catalog(), current_schema();

##Cadastro
###Ingestão - cadastro_beneficiarios
Origem:
- '/Volumes/healthcare_dev/landing/raw/cadastrp_beneficiarios.csv'

Estratégia:
- Criação de tabela Delta
- Ingestão via COPY INTO

Descrição técnica:
- Todos os campos são mantidos como STRING na bronze
- Tipagem será aplicada na silver

In [0]:
%sql
CREATE OR REPLACE TABLE healthcare_dev.bronze.cadastro_beneficiarios(
  beneficiario_id STRING,
  cpf_hash STRING,
  data_nascimento STRING,
  idade STRING,
  sexo STRING,
  uf STRING,
  municipio STRING,
  canal_aquisicao STRING,
  segmento_vinculo STRING
)
USING DELTA;

##COPY INTO - cadastro

In [0]:
%sql
COPY INTO healthcare_dev.bronze.cadastro_beneficiarios
FROM '/Volumes/healthcare_dev/landing/raw/cadastro_beneficiarios.csv'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true')
COPY_OPTIONS ('mergeSchema' = 'false');

##Contratos
### Ingestão - Contratos_planos

Origem:
'/Volumes/healthcare_dev/landing/raw/contratos_planos.csv'

Estratégica:
- Criação da tabela
- Ingestão via COPY INTO

Objetivo:
- Base principal para métricas de churn
- Suporte a análise de vigência, inadimplência e cancelamento

In [0]:
%sql
CREATE OR REPLACE TABLE healthcare_dev.bronze.contratos_planos(
  contrato_id STRING,
  beneficiario_id STRING,
  tipo_plano STRING,
  data_inicio_vigencia STRING,
  data_fim_vigencia STRING,
  valor_mensal STRING,
  coparticipacao_flag STRING
)
USING DELTA;

##COPY INTO - Contratos_planos

In [0]:
%sql
COPY INTO healthcare_dev.bronze.contratos_planos
FROM '/Volumes/healthcare_dev/landing/raw/contratos_planos.csv'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header'='true')
COPY_OPTIONS ('mergeSchema'='false');

##SAC/SRP/Manifestações
### Ingestão - sa_srp_manifestacoes

origem:
- arquivo bruto de Excel armazenado em landing/raw
- arquivo CSV convertido armazenado em landing/stage

Estratégia:
- O Excel é mantido como raw para auditoria
- A ingestão bronze utiliza o CSV de staging

In [0]:
%sql
CREATE OR REPLACE TABLE healthcare_dev.bronze.sac_srp_manifestacoes (
  protocolo_id STRING,
  beneficiario_id STRING,
  data_abertura STRING,
  canal STRING,
  categoria STRING,
  status STRING,
  sla_horas STRING,
  tempo_resolucao_horas STRING,
  csat STRING,
  nps STRING
)
USING DELTA;

##COPY INTO - SAC/SRP/Manifestacoes

In [0]:
%sql
COPY INTO healthcare_dev.bronze.sac_srp_manifestacoes
FROM '/Volumes/healthcare_dev/landing/stage/sac_srp_manifestacoes.csv'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header'='true')
COPY_OPTIONS ('mergeSchema'='false');

## Faturas - JSONL
###Ingestão - faturas_pagamentos

Origem:
- '/Volumes/healthcare_dev/landing/raw/faturas_pagamentos.jsonl'

Estratégia:
- leitura via PySpark
- cast de todas as colunas para STRING
- gravação em Delta

Motivo:
- maior previsibilidade para JSONL
- evita conflitos de inferência de schema no Bronze


In [0]:
%python
spark.sql("USE CATALOG healthcare_dev")
spark.sql("USE SCHEMA bronze")

path = "/Volumes/healthcare_dev/landing/raw/faturas_pagamentos.jsonl"

df = spark.read.json(path)

for c in df.columns:
    df = df.withColumn(c, df[c].cast("string"))

(
    df.write
      .format("delta")
      .mode("overwrite")
      .option("overwriteSchema", "true")
      .saveAsTable("healthcare_dev.bronze.faturas_pagamentos")
)

## LOGS - JSONL
### Ingestão — app_event_log

Origem:
- '/Volumes/healthcare_dev/landing/raw/app_event_log.jsonl'

Estratégia:
- leitura via PySpark
- normalização de schema em STRING
- gravação em Delta


In [0]:
%python
spark.sql("USE CATALOG healthcare_dev")
spark.sql("USE SCHEMA bronze")

path = "/Volumes/healthcare_dev/landing/raw/app_event_log.jsonl"

df = spark.read.json(path)

for c in df.columns:
    df = df.withColumn(c, df[c].cast("string"))

(
    df.write
      .format("delta")
      .mode("overwrite")
      .option("overwriteSchema", "true")
      .saveAsTable("healthcare_dev.bronze.app_event_log")
)


In [0]:
%sql
DESCRIBE TABLE healthcare_dev.bronze.app_event_log;

##Eventos - Parquet
### Ingestão — eventos_assistenciais

Origem:
- '/Volumes/healthcare_dev/landing/raw/eventos_assistenciais_compat.parquet'

Estratégia:
- leitura via PySpark
- gravação em Delta

Observação:
- o arquivo parquet utilizado nesta etapa já deve estar compatível com o runtime do Spark
- a normalização do timestamp foi tratada previamente na geração/staging


In [0]:
%python
spark.sql("USE CATALOG healthcare_dev")
spark.sql("USE SCHEMA bronze")

path = "/Volumes/healthcare_dev/landing/raw/eventos_assistenciais.parquet"

df = spark.read.parquet(path)

(
    df.write
      .format("delta")
      .mode("overwrite")
      .option("overwriteSchema", "true")
      .saveAsTable("healthcare_dev.bronze.eventos_assistenciais")
)

##Valida Contagens
Após a ingestão, valida se todas as tabelas foram criadas e se os volumes carregados correspondem ao esperado.


In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM healthcare_dev.bronze.cadastro_beneficiarios) AS n_benef,
  (SELECT COUNT(*) FROM healthcare_dev.bronze.contratos_planos)       AS n_contratos,
  (SELECT COUNT(*) FROM healthcare_dev.bronze.eventos_assistenciais)  AS n_eventos,
  (SELECT COUNT(*) FROM healthcare_dev.bronze.faturas_pagamentos)     AS n_faturas,
  (SELECT COUNT(*) FROM healthcare_dev.bronze.sac_srp_manifestacoes)  AS n_sac,
  (SELECT COUNT(*) FROM healthcare_dev.bronze.app_event_log)          AS n_logs;


##Quality check bronze
A camada Bronze não corrige dados, mas deve preservar anomalias de origem.

Nesta etapa verifica se os dados sujos gerados propositalmente continuam presentes, garantindo que a camada silver tenha material para aplicar regras de qualidade.


###Check financeiro eventos

In [0]:
%sql
SELECT COUNT(*) AS qtd_valor_pago_negativo
FROM healthcare_dev.bronze.eventos_assistenciais
WHERE valor_pago < 0;

###Check temporal contratos

In [0]:
%sql
SELECT COUNT(*) AS qtd_datas_invalidas
FROM healthcare_dev.bronze.contratos_planos
WHERE data_fim_vigencia < data_inicio_vigencia;

###Check proporção de valores negativos

In [0]:
%sql
SELECT
  COUNT(*) AS total,
  SUM(CASE WHEN valor_pago < 0 THEN 1 ELSE 0 END) AS qtd_negativos,
  ROUND(100.0 * SUM(CASE WHEN valor_pago < 0 THEN 1 ELSE 0 END) / COUNT(*), 4) AS pct_negativos
FROM healthcare_dev.bronze.eventos_assistenciais;

##Adiciona colunas de auditoria

In [0]:
%sql
ALTER TABLE healthcare_dev.bronze.cadastro_beneficiarios
ADD COLUMNS (ingestion_ts TIMESTAMP, load_id STRING, source_file STRING);

ALTER TABLE healthcare_dev.bronze.contratos_planos
ADD COLUMNS (ingestion_ts TIMESTAMP, load_id STRING, source_file STRING);

ALTER TABLE healthcare_dev.bronze.sac_srp_manifestacoes
ADD COLUMNS (ingestion_ts TIMESTAMP, load_id STRING, source_file STRING);

ALTER TABLE healthcare_dev.bronze.faturas_pagamentos
ADD COLUMNS (ingestion_ts TIMESTAMP, load_id STRING, source_file STRING);

ALTER TABLE healthcare_dev.bronze.app_event_log
ADD COLUMNS (ingestion_ts TIMESTAMP, load_id STRING, source_file STRING);

ALTER TABLE healthcare_dev.bronze.eventos_assistenciais
ADD COLUMNS (ingestion_ts TIMESTAMP, load_id STRING, source_file STRING);

##Preenche ingestion_ts e load_id onde está nulo

In [0]:
%sql
UPDATE healthcare_dev.bronze.cadastro_beneficiarios
SET ingestion_ts = current_timestamp(),
    load_id = 'backfill_001',
    source_file = '/Volumes/healthcare_dev/landing/raw/cadastro_beneficiarios.csv'
WHERE ingestion_ts IS NULL;

UPDATE healthcare_dev.bronze.contratos_planos
SET ingestion_ts = current_timestamp(),
    load_id = 'backfill_001',
    source_file = '/Volumes/healthcare_dev/landing/raw/contratos_planos.csv'
WHERE ingestion_ts IS NULL;

UPDATE healthcare_dev.bronze.sac_srp_manifestacoes
SET ingestion_ts = current_timestamp(),
    load_id = 'backfill_001',
    source_file = '/Volumes/healthcare_dev/landing/stage/sac_srp_manifestacoes.csv'
WHERE ingestion_ts IS NULL;

UPDATE healthcare_dev.bronze.faturas_pagamentos
SET ingestion_ts = current_timestamp(),
    load_id = 'backfill_001',
    source_file = '/Volumes/healthcare_dev/landing/raw/faturas_pagamentos.jsonl'
WHERE ingestion_ts IS NULL;

UPDATE healthcare_dev.bronze.app_event_log
SET ingestion_ts = current_timestamp(),
    load_id = 'backfill_001',
    source_file = '/Volumes/healthcare_dev/landing/raw/app_event_log.jsonl'
WHERE ingestion_ts IS NULL;

UPDATE healthcare_dev.bronze.eventos_assistenciais
SET ingestion_ts = current_timestamp(),
    load_id = 'backfill_001',
    source_file = '/Volumes/healthcare_dev/landing/raw/eventos_assistenciais.parquet'
WHERE ingestion_ts IS NULL;